# Crafting an AI-Powered HR Assistant

## Project Overview
This project builds a **conversational chatbot** that answers queries about Nestlé's HR policy document. It leverages:
- **PyPDFLoader** – to extract text from the HR policy PDF
- **OpenAI Embeddings** – to convert text chunks into numerical vectors
- **ChromaDB** – as a vector store for efficient similarity search
- **GPT-3.5 Turbo** – as the language model to generate accurate answers
- **Gradio** – to provide a user-friendly chatbot interface

**Dataset:** `Dataset/the_nestle_hr_policy_pdf_2012.pdf`

---

## Situation
As a developer tasked with improving Nestlé HR department efficiency, this chatbot allows HR staff and employees to query Nestlé's HR policy document conversationally, getting instant, accurate answers without manually searching through the document.

## Step 1: Install Required Libraries

In [ ]:
%pip install openai langchain langchain-openai langchain-community langchain-chroma \
             chromadb pypdf gradio tiktoken

## Step 2: Import Essential Tools and Set Up OpenAI API Environment

In [ ]:
import os
import gradio as gr

# LangChain document loaders and text splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector store and embeddings
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# QA chain components
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

print("All libraries imported successfully!")

# -------------------------------------------------------
# Configure OpenAI API Key
# Set the environment variable before running:
#   export OPENAI_API_KEY="your-api-key-here"
# -------------------------------------------------------
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    api_key = input("Enter your OpenAI API key: ").strip()
    os.environ["OPENAI_API_KEY"] = api_key

print("OpenAI API key configured successfully!")

## Step 3: Load and Split the Nestlé HR Policy PDF

Use **PyPDFLoader** to load the document, then split it into manageable chunks using **RecursiveCharacterTextSplitter** for efficient processing and retrieval.

In [ ]:
# Path to the Nestlé HR policy PDF
PDF_PATH = "Dataset/the_nestle_hr_policy_pdf_2012.pdf"

# Load the PDF using PyPDFLoader
print(f"Loading PDF from: {PDF_PATH}")
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\nSample content from page 1:\n{documents[0].page_content[:500]}...")

In [ ]:
# Split documents into smaller chunks for processing
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Maximum characters per chunk
    chunk_overlap=200,     # Overlap between chunks to preserve context
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

text_chunks = text_splitter.split_documents(documents)

print(f"Total text chunks created: {len(text_chunks)}")
print(f"\nSample chunk:\n{text_chunks[0].page_content}")
print(f"\nChunk metadata: {text_chunks[0].metadata}")

## Step 4: Create Vector Representations Using ChromaDB and OpenAI Embeddings

Convert each text chunk into a numerical vector using **OpenAI's text-embedding model** and store them in **ChromaDB** for fast similarity-based retrieval.

In [ ]:
# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"  # OpenAI's efficient embedding model
)

# Create ChromaDB vector store from the text chunks
print("Creating vector store from text chunks...")
print("This may take a moment as embeddings are generated for each chunk...")

vector_store = Chroma.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policy",
    persist_directory="./chroma_db"   # Persist to disk for reuse
)

print(f"\nVector store created successfully!")
print(f"Total vectors stored: {vector_store._collection.count()}")

# Set up the retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # Retrieve top 5 most relevant chunks
)

print("Retriever configured successfully!")

## Step 5: Build the Question-Answering System with GPT-3.5 Turbo

Create a **RetrievalQA chain** powered by GPT-3.5 Turbo that retrieves relevant text chunks from ChromaDB and uses them as context to generate accurate answers.

In [ ]:
# Initialize the GPT-3.5 Turbo language model
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.2,      # Low temperature for factual, consistent answers
    max_tokens=1024
)

print("GPT-3.5 Turbo model initialized!")

## Step 6: Create a Prompt Template

Design a structured **prompt template** that guides the chatbot to stay focused on Nestlé's HR policy content and communicate clearly with users.

In [ ]:
# Define the prompt template for the HR chatbot
prompt_template = """You are a knowledgeable and professional HR assistant for Nestlé. 
Your role is to help employees and HR staff by answering questions about Nestlé's HR policies 
accurately and clearly based on the provided context.

Guidelines:
- Answer questions ONLY based on the context provided from the Nestlé HR policy document.
- If the answer is not found in the context, respond: "I'm sorry, I couldn't find that information in the Nestlé HR policy document. Please consult your HR department for further assistance."
- Be professional, concise, and helpful in your responses.
- When relevant, mention the specific policy area or section your answer relates to.

Context from Nestlé HR Policy:
{context}

Employee Question: {question}

HR Assistant Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully!")
print("\nTemplate variables:", PROMPT.input_variables)

In [ ]:
# Build the RetrievalQA chain combining the LLM, retriever, and prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # "stuff" passes all retrieved docs into the prompt
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("QA chain built successfully!")

## Step 7: Test the QA System

Run a quick test query to verify the pipeline works before building the Gradio interface.

In [ ]:
def ask_question(query: str) -> str:
    """Query the QA chain and return the answer."""
    result = qa_chain.invoke({"query": query})
    return result["result"]


# Test with a sample HR policy question
test_query = "What are Nestlé's core principles regarding employee development and training?"
print(f"Test Query: {test_query}\n")
print("Answer:")
print("-" * 60)
answer = ask_question(test_query)
print(answer)

## Step 8: Build and Launch the Gradio Chatbot Interface

Create a user-friendly **Gradio chatbot interface** with:
- A chat window showing conversation history
- A text input for questions
- Example questions to help users get started
- A clear button to reset the conversation

In [ ]:
def chatbot_response(message: str, history: list) -> tuple:
    """
    Process a user message and return the chatbot response.

    Args:
        message (str): The user's question.
        history (list): The conversation history as list of [user, bot] pairs.

    Returns:
        tuple: Empty string (clears input box), updated history with the new exchange.
    """
    if not message or message.strip() == "":
        return "", history

    # Query the QA chain
    response = ask_question(message.strip())

    # Append the exchange to history
    history.append([message, response])

    return "", history


# Example questions users can click to try
example_questions = [
    "What is Nestlé's policy on employee recruitment and selection?",
    "How does Nestlé support employee learning and development?",
    "What are the key principles of Nestlé's compensation and benefits policy?",
    "What does Nestlé's HR policy say about diversity and inclusion?",
    "What are Nestlé's guidelines for performance management and appraisal?",
    "How does Nestlé handle employee health and safety in the workplace?"
]

print("Chatbot response function defined!")

In [ ]:
# Build the Gradio chatbot interface
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Nestlé HR Policy Assistant"
) as demo:

    # Header
    gr.Markdown("""
    # Nestlé HR Policy Assistant
    ### Powered by OpenAI GPT-3.5 Turbo · ChromaDB · LangChain · Gradio
    Ask any question about **Nestlé's HR Policy (2012)** and get instant, accurate answers.
    """)

    # Chatbot display
    chatbot = gr.Chatbot(
        label="Conversation",
        height=450,
        bubble_full_width=False,
        show_label=True
    )

    # Input row
    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Ask a question about Nestlé's HR policies...",
            label="Your Question",
            lines=2,
            scale=9
        )
        submit_btn = gr.Button("Ask", variant="primary", scale=1)

    # Action buttons
    with gr.Row():
        clear_btn = gr.Button("Clear Conversation", variant="secondary")

    # Example questions
    gr.Markdown("### Example Questions")
    gr.Examples(
        examples=example_questions,
        inputs=user_input,
        label="Click an example to populate the question box"
    )

    # Footer
    gr.Markdown("""
    ---
    *This assistant answers based solely on the Nestlé HR Policy document (2012).
    For official HR guidance, please consult Nestlé's HR department.*
    """)

    # Wire up events
    submit_btn.click(
        fn=chatbot_response,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot]
    )
    user_input.submit(
        fn=chatbot_response,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot]
    )
    clear_btn.click(
        fn=lambda: ([], ""),
        inputs=None,
        outputs=[chatbot, user_input]
    )

# Launch the interface
demo.launch(share=False)

## Result & Summary

This notebook delivers a complete end-to-end **AI-Powered HR Assistant** for Nestlé:

| Component | Tool Used | Purpose |
|-----------|-----------|----------|
| **PDF Loading** | `PyPDFLoader` | Extract text from Nestlé HR policy PDF |
| **Text Chunking** | `RecursiveCharacterTextSplitter` | Split document into overlapping chunks |
| **Embeddings** | `OpenAIEmbeddings` (ada-002) | Convert text chunks to numerical vectors |
| **Vector Store** | `ChromaDB` | Store and retrieve vectors by similarity |
| **Language Model** | `GPT-3.5 Turbo` | Generate accurate, context-aware answers |
| **Prompt Engineering** | `PromptTemplate` | Guide chatbot behavior and response style |
| **QA Pipeline** | `RetrievalQA` | Combine retrieval + generation into one chain |
| **User Interface** | `Gradio` | Conversational chatbot web interface |

### Key Outcomes
- Employees can query Nestlé's HR policy in natural language without manual document search
- The RAG (Retrieval-Augmented Generation) pipeline ensures answers are grounded in actual policy content
- ChromaDB persists the vector store, enabling faster subsequent launches
- The Gradio interface is accessible via browser with no additional setup

### Potential Enhancements
- Add support for multiple HR policy documents
- Implement conversation memory for multi-turn follow-up questions
- Add source citation showing which page/section the answer came from
- Deploy to a cloud platform (AWS, GCP, Hugging Face Spaces) for team-wide access